# CIS 536/736 - Week 5 / Module 4: Procedural Textures & Shader Graph
### Lab 2b & MP 3 Preparation

**Objective:** In this notebook, we bridge the gap between procedural shader theory and engine application. You will prepare your Unity 6 HDRP environment, validate your OpenUSD stage meshes, and generate the C# wrapper scripts required to drive your Shader Graph Blackboard parameters dynamically for Machine Problem 3 (MP 3).

**Core Concepts Covered:**
- Escaping 2D UV mapping via Procedural Noise (Voronoi/Perlin) and Triplanar Projection.
- Modifying the Fragment Context (Emission/Color) and Vertex Context (Displacement).
- Exposing data streams (Time, Vectors) to the Blackboard.
- Programmatic interaction with OpenUSD assets.

## Part 1: OpenUSD Stage Validation (Python)

Before we write vertex displacement shaders, we must ensure our imported OpenUSD stage actually contains the required sub-meshes. We will use the `pxr` USD core library to traverse the stage hierarchy and verify target primitives.

In [ ]:
# Install the USD core library if missing: !pip install usd-core
from pxr import Usd, UsdGeom

def validate_usd_stage(stage_path: str):
    print(f"Opening USD Stage: {stage_path}")
    try:
        stage = Usd.Stage.Open(stage_path)
        if not stage:
            print("Error: Could not open stage.")
            return
            
        mesh_count = 0
        for prim in stage.Traverse():
            if prim.IsA(UsdGeom.Mesh):
                mesh_count += 1
                print(f"[VALID] Found Mesh Geometry: {prim.GetPath()}")
        
        if mesh_count == 0:
            print("WARNING: No mesh geometry found. Vertex displacement shaders will have no effect.")
        else:
            print(f"\nSUCCESS: Found {mesh_count} valid meshes ready for Shader Graph assignment.")
            
    except Exception as e:
        print(f"Validation Failed: {e}")

# Example usage (Replace with your MP3 USD asset path)
# validate_usd_stage('Assets/Models/Environment_Stage.usd')

## Part 2: Shader Graph Pipeline Setup (Unity 6)

### Step 2.1: Initialize the Graph
1. In Unity, right-click the Project window -> **Create > Shader Graph > HDRP > Lit Shader Graph**.
2. Name it `Procedural_Displacement_Mat`.
3. Double-click to open the visual canvas.

### Step 2.2: The Blackboard Parameters
Open the Blackboard (top left) and expose the following properties for MP 3:
- `BaseColor` (Type: Color, Default: White)
- `NoiseScale` (Type: Float, Default: 10.0)
- `AnimationSpeed` (Type: Float, Default: 1.0)
- `DisplacementStrength` (Type: Float, Default: 0.5)

### Step 2.3: Procedural Noise & Time (The Logic)
1. Create a **Time** node. Extract the `Time` output.
2. Multiply `Time` by your `AnimationSpeed` property.
3. Feed the result into a **Tiling And Offset** node (Offset port).
4. Feed the UV output into a **Voronoi** or **Simple Noise** node.
5. Plug your `NoiseScale` property into the Scale port of the noise node.

### Step 2.4: Vertex Displacement
1. Take the output of your Noise node and multiply it by `DisplacementStrength`.
2. Create a **Normal Vector** node (Space: Object).
3. Multiply the scaled noise by the Normal Vector.
4. Create a **Position** node (Space: Object). Add the modified normal vector to the Position.
5. Pipe the result into the **Position** port of the **Vertex Context** in the Master Stack.

## Part 3: MP 3 C# Parameterization Generator

For Machine Problem 3, you are required to dynamically manipulate these shader parameters at runtime via a MonoBehaviour script. The Python script below will automatically generate the boilerplate C# code needed to interface with your Blackboard properties.

In [ ]:
def generate_shader_controller_script(script_name="ShaderGraphController"):
    csharp_code = f"""using UnityEngine;

public class {script_name} : MonoBehaviour
{{
    [Header("Material References")]
    public Material proceduralMaterial;

    [Header("Dynamic Shader Parameters")]
    [Range(0f, 50f)] public float noiseScale = 10f;
    [Range(0f, 5f)] public float animationSpeed = 1f;
    [Range(0f, 2f)] public float displacementStrength = 0.5f;

    // Shader Property IDs (Performance Optimization)
    private int noiseScaleId;
    private int animationSpeedId;
    private int displacementStrengthId;

    void Start()
    {{
        if (proceduralMaterial == null)
        {{
            Debug.LogError("Procedural Material not assigned!");
            enabled = false;
            return;
        }}

        // Cache string lookups into integer hashes for performance
        noiseScaleId = Shader.PropertyToID("_NoiseScale");
        animationSpeedId = Shader.PropertyToID("_AnimationSpeed");
        displacementStrengthId = Shader.PropertyToID("_DisplacementStrength");
    }}

    void Update()
    {{
        // Send real-time updates from CPU to the GPU Shader
        proceduralMaterial.SetFloat(noiseScaleId, noiseScale);
        proceduralMaterial.SetFloat(animationSpeedId, animationSpeed);
        proceduralMaterial.SetFloat(displacementStrengthId, displacementStrength);
        
        // MP3 requirement: Oscillate the displacement dynamically over time
        // float dynamicDisplacement = Mathf.PingPong(Time.time, displacementStrength);
        // proceduralMaterial.SetFloat(displacementStrengthId, dynamicDisplacement);
    }}
}}
"""
    print(f"// Save this output to {script_name}.cs in Unity\n")
    print(csharp_code)

generate_shader_controller_script()

## 🚀 Machine Problem 3 (MP 3) Deliverable Checklist

1. **Graph Compilation:** Successfully compile the HDRP Shader Graph with 0 errors.
2. **OpenUSD Application:** Create a Material instance of the shader and assign it to an imported `.usd` stage asset in the scene.
3. **Vertex Displacement:** The geometry must physically deform using a noise function piped into the Vertex Position node.
4. **C# Controller:** Attach the generated `ShaderGraphController.cs` script to an empty GameObject, assign your material, and manipulate the sliders in Play Mode to alter the shader in real-time.
5. **Submission:** Push the updated Unity Project (excluding `Library/` and `Temp/`) to your `cis536_736_hw` GitHub repository before the midnight Portcullis.